# Proyecto de Recuperación de Información


## Análisis y Procesamiento de Corpus de Documentos
- Carga de datos
- Limpieza y exploración
- Preprocesamiento (tokenización, eliminación de stopwords, normalización, stemming)


In [1]:
# Instalación de dependencias
#!pip install nltk scikit-learn numpy pandas kagglehub

### Sección 1: Configuración e Instalación


In [32]:
from pathlib import Path
import pandas as pd
import unicodedata
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [33]:
# Descargar recursos de NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

True

### Sección 2: Carga de Datos


In [34]:
# Descarga del dataset (comentado si ya existe)
import kagglehub

# Download latest version into ./data
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
path = kagglehub.dataset_download(
    "thedevastator/uncovering-financial-insights-with-the-reuters-2",
    output_dir=str(data_dir),
)

print("Path to dataset files:", path)

Path to dataset files: data


In [42]:
for file in data_dir.iterdir():
    print(f"  - {file.name}")


  - .complete
  - ModApte_test.csv
  - ModApte_train.csv
  - ModApte_unused.csv
  - ModHayes_test.csv
  - ModHayes_train.csv
  - ModLewis_test.csv
  - ModLewis_train.csv
  - ModLewis_unused.csv


In [43]:
import os
import csv  # <--- CORRECCIÓN: Aquí importamos la librería que faltaba

def load_all_corpus_csv(corpus_path):
    """
    Escanea la carpeta data y lee de forma masiva todos los archivos CSV fila por fila.
    """
    all_rows = []
    if not os.path.exists(corpus_path):
        print(f"Error: La ruta '{corpus_path}' no existe.")
        return pd.DataFrame(columns=["doc_id", "title", "text"])

    for file_name in os.listdir(corpus_path):
        # Buscamos todos los archivos .csv sueltos en la carpeta data
        if file_name.lower().endswith('.csv'):
            file_path = os.path.join(corpus_path, file_name)
            dataset_name = os.path.splitext(file_name)[0]

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    reader = csv.reader(f)
                    for i, row in enumerate(reader):
                        if not row or len(row) == 0:
                            continue
                        # ID único para cada documento combinando el dataset y la fila
                        doc_id = f"{dataset_name}_doc_{i+1}"
                        text_content = " ".join(row)
                        # Estructuramos en columnas compatibles con tu limpieza de datos
                        all_rows.append({"doc_id": doc_id, "title": "", "text": text_content})
            except Exception as e:
                print(f"Error en archivo {file_name}: {e}")

    df = pd.DataFrame(all_rows)
    print("=" * 100)
    print(f"Dimensión total del dataset completo: {df.shape}")
    print("=" * 100)
    return df

# CORRECCIÓN DE RUTA: Como tus archivos están directo en 'data', apuntamos a esa carpeta
ruta_corpus = 'data'
corpus = load_all_corpus_csv(ruta_corpus)

Dimensión total del dataset completo: (55745, 3)


### Sección 3: Exploración de Datos

In [44]:
print("Vista previa de los datos:")
corpus.head(10)

Vista previa de los datos:


,doc_id,title,text
0,ModApte_test_doc_1,,text text_type topics lewis_split cgis_split o...
1,ModApte_test_doc_2,,Mounting trade friction between the\nU.S. And ...
2,ModApte_test_doc_3,,A survey of 19 provinces and seven cities\nsho...
3,ModApte_test_doc_4,,The Ministry of International Trade and\nIndus...
4,ModApte_test_doc_5,,Thailand's trade deficit widened to 4.5\nbilli...
5,ModApte_test_doc_6,,Indonesia expects crude palm oil (CPO)\nprices...
6,ModApte_test_doc_7,,"""BRIEF"" [] ""TEST"" ""TRAINING-SET"" ""3818"" ""1483..."
7,ModApte_test_doc_8,,"Tug crews in New South Wales (NSW),\nVictoria ..."
8,ModApte_test_doc_9,,The Indonesian Commodity Exchange is\nlikely t...
9,ModApte_test_doc_10,,Food Department officials said the U.S.\nDepar...


## Sección 4: Limpieza de Datos

In [45]:
def clean_dataset(df):
    """
    Limpia el dataset: rellena valores nulos, limpia columnas y crea campo de documento.
    """

    df = df.copy()

    # Completar valores nulos
    df["title"] = df["title"].fillna("")
    df["text"] = df["text"].fillna("")

    # Limpiar columnas innecesarias
    cols = ["text_type", "lewis_split", "cgis_split"]

    for col in cols:

        if col in df.columns:

            df[col] = (
                df[col]
                .astype(str)
                .str.replace('"', '', regex=False)
            )

    # Crear documento combinado
    df["document"] = (
        df["title"].astype(str)
        + " " +
        df["text"].astype(str)
    )

    # Limpiar espacios extras
    df["document"] = (
        df["document"]
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    # Eliminar documentos vacíos
    df = df[
        df["document"].str.strip() != ""
    ]

    print(f"Nuevo tamaño del dataset: {df.shape}")

    return df

In [46]:
corpus = clean_dataset(corpus)


Nuevo tamaño del dataset: (55745, 4)


In [47]:
print("Dataset limpio - Primeras filas:")
corpus.head(10)

Dataset limpio - Primeras filas:


,doc_id,title,text,document
0,ModApte_test_doc_1,,text text_type topics lewis_split cgis_split o...,text text_type topics lewis_split cgis_split o...
1,ModApte_test_doc_2,,Mounting trade friction between the\nU.S. And ...,Mounting trade friction between the U.S. And J...
2,ModApte_test_doc_3,,A survey of 19 provinces and seven cities\nsho...,A survey of 19 provinces and seven cities show...
3,ModApte_test_doc_4,,The Ministry of International Trade and\nIndus...,The Ministry of International Trade and Indust...
4,ModApte_test_doc_5,,Thailand's trade deficit widened to 4.5\nbilli...,Thailand's trade deficit widened to 4.5 billio...
5,ModApte_test_doc_6,,Indonesia expects crude palm oil (CPO)\nprices...,Indonesia expects crude palm oil (CPO) prices ...
6,ModApte_test_doc_7,,"""BRIEF"" [] ""TEST"" ""TRAINING-SET"" ""3818"" ""1483...","""BRIEF"" [] ""TEST"" ""TRAINING-SET"" ""3818"" ""14835..."
7,ModApte_test_doc_8,,"Tug crews in New South Wales (NSW),\nVictoria ...","Tug crews in New South Wales (NSW), Victoria a..."
8,ModApte_test_doc_9,,The Indonesian Commodity Exchange is\nlikely t...,The Indonesian Commodity Exchange is likely to...
9,ModApte_test_doc_10,,Food Department officials said the U.S.\nDepar...,Food Department officials said the U.S. Depart...


### Sección 5: Pipeline de Preprocesamiento

El preprocesamiento incluye:
1. **Tokenización**: Dividir documentos en palabras
2. **Eliminación de stopwords**: Remover palabras comunes sin valor semántico
3. **Normalización**: Convertir a minúsculas, quitar acentos, eliminar caracteres especiales
4. **Stemming**: Reducir palabras a su raíz

#### 5.1 Tokenización

In [48]:
def tokenize(corpus):
    """
    Tokeniza cada documento en el corpus en palabras individuales.

    Args:
        corpus (pd.DataFrame): DataFrame con documentos

    Returns:
        pd.DataFrame: DataFrame con columna 'tokens'
    """
    # Tokenizar cada documento
    corpus["tokens"] = (corpus["document"].apply(word_tokenize))

    print(f"Tokens creados. Ejemplo (primeros 10): {corpus['tokens'].iloc[0][:10]}")
    return corpus


#### 5.2 Eliminación de Stopwords

In [49]:
def quit_stopwords(corpus):
    """
    Elimina stopwords (palabras comunes) del corpus y convierte a minúsculas.

    Args:
        corpus (pd.DataFrame): DataFrame con tokens

    Returns:
        pd.DataFrame: DataFrame con columna 'no_stopwords'
    """
    stop_words = set(stopwords.words("english"))

    # Poner en minúsculas
    corpus["no_stopwords"] = (
        corpus["tokens"]
        .apply(
            lambda words: [
                word.lower()
                for word in words
            ]
        )
    )

    # Quitar las stopwords
    corpus["no_stopwords"] = (
        corpus["no_stopwords"]
        .apply(
            lambda words: [
                word
                for word in words
                if word not in stop_words
            ]
        )
    )

    print(f"Stopwords removidos. Ejemplo: {corpus['no_stopwords'].iloc[0][:10]}")
    return corpus


#### 5.3 Normalización

In [50]:
def normalize(tokens):
    """
    Normaliza tokens: convierte a minúsculas, elimina acentos,
    y mantiene solo caracteres alfabéticos.

    Args:
        tokens (list): Lista de tokens

    Returns:
        list: Lista de tokens normalizados
    """
    normalized = []

    for token in tokens:
        # lowercase (case folding)
        token = token.lower()

        # quitar acentos
        token = unicodedata.normalize(
            "NFKD",
            token
        ).encode(
            "ascii",
            "ignore"
        ).decode("utf-8")

        # dejar solo letras
        token = re.sub(
            r"[^a-z]",
            "",
            token
        )

        # evitar vacíos
        if token:
            normalized.append(token)

    return normalized


#### 5.4 Stemming

In [51]:
def stemming(corpus):
    """
    Aplica stemming (reducción a raíz) a los tokens normalizados.

    Args:
        corpus (pd.DataFrame): DataFrame con columna 'normalized'

    Returns:
        pd.DataFrame: DataFrame con columna 'stemmed'
    """
    stemmer = PorterStemmer()

    corpus["stemmed"] = (
        corpus["normalized"]
        .apply(
            lambda words: [
                stemmer.stem(word)
                for word in words
            ]
        )
    )

    print(f"Stemming aplicado. Ejemplo: {corpus['stemmed'].iloc[0][:10]}")
    return corpus

#### 5.5 Pipeline Completo de Preprocesamiento

In [52]:
def preprocessing(corpus):
    """
    Ejecuta el pipeline completo de preprocesamiento:
    tokenización → eliminación de stopwords → normalización → stemming

    Args:
        corpus (pd.DataFrame): DataFrame con documentos originales

    Returns:
        pd.DataFrame: DataFrame preprocesado con columnas de tokens, stopwords, normalized, y stemmed
    """
    print("Iniciando pipeline de preprocesamiento...\n")

    # Tokenizar
    print("1. Tokenización...")
    corpus = tokenize(corpus)

    # Quitar stopwords
    print("\n2. Eliminación de stopwords...")
    corpus = quit_stopwords(corpus)

    # Normalización (case folding, acentos, etc.)
    print("\n3. Normalización...")
    corpus["normalized"] = (corpus["no_stopwords"].apply(normalize))

    # Stemming
    print("\n4. Stemming...")
    corpus = stemming(corpus)

    print("\nPipeline completado exitosamente")
    return corpus


### Sección 6: Ejecución del Pipeline

In [53]:
print("Procesando corpus...")
corpus = preprocessing(corpus)

Procesando corpus...
Iniciando pipeline de preprocesamiento...

1. Tokenización...
Tokens creados. Ejemplo (primeros 10): ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs']

2. Eliminación de stopwords...
Stopwords removidos. Ejemplo: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs']

3. Normalización...

4. Stemming...
Stemming aplicado. Ejemplo: ['text', 'texttyp', 'topic', 'lewissplit', 'cgissplit', 'oldid', 'newid', 'place', 'peopl', 'org']

Pipeline completado exitosamente


### Sección 7: Resultados

In [54]:
print("Corpus preprocesado - Información final:")
print(f"Tamaño del corpus: {corpus.shape}")
print(f"\nColumnas disponibles: {list(corpus.columns)}")

Corpus preprocesado - Información final:
Tamaño del corpus: (55745, 8)

Columnas disponibles: ['doc_id', 'title', 'text', 'document', 'tokens', 'no_stopwords', 'normalized', 'stemmed']


In [55]:
print("Ejemplo de tokens procesados (primeros 100 elementos del primer documento):")
print(corpus["stemmed"].iloc[0][:100])

Ejemplo de tokens procesados (primeros 100 elementos del primer documento):
['text', 'texttyp', 'topic', 'lewissplit', 'cgissplit', 'oldid', 'newid', 'place', 'peopl', 'org', 'exchang', 'date', 'titl']


In [56]:
print("\nEstadísticas de tokenización y preprocesamiento:")
print(f"Promedio de tokens por documento: {corpus['tokens'].apply(len).mean():.2f}")
print(f"Promedio de tokens sin stopwords: {corpus['no_stopwords'].apply(len).mean():.2f}")
print(f"Promedio de tokens finales (después de stemming): {corpus['stemmed'].apply(len).mean():.2f}")



Estadísticas de tokenización y preprocesamiento:
Promedio de tokens por documento: 177.84
Promedio de tokens sin stopwords: 135.34
Promedio de tokens finales (después de stemming): 88.58


In [57]:
corpus[["document", "stemmed"]].head(5)

,document,stemmed
0,text text_type topics lewis_split cgis_split o...,"[text, texttyp, topic, lewissplit, cgissplit, ..."
1,Mounting trade friction between the U.S. And J...,"[mount, trade, friction, us, japan, rais, fear..."
2,A survey of 19 provinces and seven cities show...,"[survey, provinc, seven, citi, show, vermin, c..."
3,The Ministry of International Trade and Indust...,"[ministri, intern, trade, industri, miti, revi..."
4,Thailand's trade deficit widened to 4.5 billio...,"[thailand, s, trade, deficit, widen, billion, ..."


## HASTA AQUI SE FUE LA EJECUCION 

## DE AQUI FALTA EJECUTAR LAS DEMAS CELDAS POR SI ACASO

-----

----

----

----

## Construcción del índice

• Construcción de un índice invertido que almacene, para cada término, los documentos en los que
aparece y su frecuencia

In [24]:
def create_inverted_index(corpus):
    inverted_index = {}

    for _, rows in corpus.iterrows():
        id_doc = rows["new_id"]
        tokens = rows["stemmed"]

        for token in tokens:

            if token not in inverted_index:
                inverted_index[token] = {}

            if id_doc not in inverted_index[token]:
                inverted_index[token][id_doc] = 1
            else:
                inverted_index[token][id_doc] += 1

    return inverted_index

In [25]:
inverted_index = create_inverted_index(corpus)

In [28]:
print(f"Indice invertido de market:{inverted_index['switzerland']}")
# print(inverted_index["stocks"])
# print(inverted_index["oil"])

Indice invertido de market:{'"10"': 1, '"179"': 3, '"204"': 1, '"359"': 1, '"394"': 1, '"415"': 1, '"419"': 1, '"427"': 1, '"458"': 1, '"468"': 1, '"501"': 2, '"545"': 1, '"759"': 1, '"797"': 1, '"1248"': 1, '"1675"': 1, '"2214"': 1, '"2257"': 1, '"2280"': 2, '"2503"': 1, '"2790"': 1, '"3192"': 1, '"3536"': 1, '"3554"': 1, '"3792"': 1, '"3844"': 1, '"3901"': 1, '"3923"': 1, '"4022"': 1, '"4638"': 1, '"4691"': 1, '"4692"': 1, '"5032"': 1, '"5813"': 1, '"5824"': 1, '"5855"': 1, '"6658"': 2, '"7035"': 1, '"7043"': 1, '"7088"': 4, '"7104"': 1, '"7175"': 1, '"7311"': 1, '"7950"': 1, '"8199"': 1, '"8421"': 1, '"8635"': 1, '"8667"': 1, '"8675"': 1, '"8855"': 1, '"9325"': 1, '"9410"': 1, '"9448"': 1, '"9570"': 1, '"9835"': 1, '"9866"': 1, '"9905"': 1, '"10428"': 1, '"10849"': 1, '"11163"': 1, '"11830"': 2, '"12136"': 1, '"12482"': 1, '"13005"': 1}


## Recuperación

### Recuperación basada en similitud Jaccard utilizando vectores binarios

In [29]:
def jaccard_similarity(set1, set2):
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))

    return intersection / union

In [30]:
def jaccard_search(query, corpus, top_k=10):

    query_terms = set(query.lower().split())

    results = []

    for _, row in corpus.iterrows():

        doc_terms = set(row["stemmed"])

        score = jaccard_similarity(
            query_terms,
            doc_terms
        )

        results.append(
            (row["new_id"], score)
        )

    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return results[:top_k]

In [31]:
results = jaccard_search("switzerland computer", corpus)
print(results)

[('"10849"', 0.045454545454545456), ('"3844"', 0.041666666666666664), ('"12482"', 0.038461538461538464), ('"2214"', 0.037037037037037035), ('"8199"', 0.037037037037037035), ('"9570"', 0.037037037037037035), ('"204"', 0.034482758620689655), ('"10428"', 0.034482758620689655), ('"14393"', 0.03333333333333333), ('"9325"', 0.029411764705882353)]
